# Stage 8: Grad-CAM and comparison with the annotated bounding boxes
**Deep Learning-Based Detection and Classification of Brain Tumors Using MRI Images**

Final candidate: ResNet18 with full fine-tuning (run R18-C). Test set only, no training here. Needs a GPU runtime (Runtime > Change runtime type > T4 GPU), about 10 minutes.

**What this notebook does**
1. Loads your saved checkpoint and checks that it reproduces the saved test predictions of R18-C (it tells you which saved run it matches).
2. Checks the Grad-CAM implementation: hooked gradients against the exact formula for this network, and the coordinate mapping from the 224 x 224 input back to the original image.
3. Computes Grad-CAM heatmaps (layer4, with layer3 as a sensitivity check) and compares them with the annotated boxes for Glioma, Meningioma and Pituitary images, against three baselines (a fixed centre blob, heatmaps taken from other images of the same class, and a random point).
4. Checks that heatmaps change when the network weights are randomised.

**What it does not show:** that the network 'looks at the tumour' for the right reasons. A heatmap is a visualisation of gradient-weighted activations, and the boxes are an imperfect reference. The layer4 map is 7 x 7, coarse compared with the tumours (median box is 1 to 3 percent of the image area).

**Inputs:** `final_results.zip` (from Stage 7), the dataset, and `R18-C_best.pt` in Google Drive under `My Drive/brain-tumor-project/models/`.

## 1. Setup and scripts

In [ ]:
import os, sys, json, zipfile, subprocess
from pathlib import Path
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'Switch to a GPU runtime: Runtime > Change runtime type > T4 GPU'
W = Path('/content/stage8'); W.mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /content/stage8/localisation.py
"""Geometry and localisation metrics for Grad-CAM heatmaps. NumPy/PIL/SciPy only (no PyTorch).

Coordinate conventions
  * An original image is W x H pixels. Preprocessing pads it with black to an s x s square, s = max(W, H),
    pasting it at (ox, oy) = ((s - W) // 2, (s - H) // 2), then resizes to 224 x 224 (same as PadToSquare in Stage 2).
  * A heatmap computed on the 224 x 224 network input is mapped back by: resize to s x s -> crop the original
    area [oy:oy+H, ox:ox+W]. After the crop it is min-max normalised inside the original image area.
  * Boxes are YOLO lines [class, cx, cy, w, h], normalised to the ORIGINAL image size (verified in Stage 1).
"""
import numpy as np
from PIL import Image
from scipy.stats import spearmanr


def pad_offset(W, H):
    s = max(W, H)
    return s, (s - W) // 2, (s - H) // 2


def to_original(cam224, W, H):
    """cam224: (224,224) float array -> (H, W) float array in original pixel coordinates, min-max normalised.
    Also returns the share of the heatmap mass that falls on the black padding (before cropping)."""
    s, ox, oy = pad_offset(W, H)
    big = np.asarray(Image.fromarray(cam224.astype(np.float32), mode='F').resize((s, s), Image.BILINEAR), dtype=np.float64)
    big = np.clip(big, 0, None)
    tot = big.sum()
    crop = big[oy:oy + H, ox:ox + W]
    pad_share = float(1 - crop.sum() / tot) if tot > 0 else float('nan')
    lo, hi = crop.min(), crop.max()
    crop = (crop - lo) / (hi - lo) if hi > lo else np.zeros_like(crop)
    return crop, pad_share


def box_mask(boxes, W, H, class_id=None):
    """Union mask of the boxes (optionally only those with the given class id). Returns (mask, n_boxes_used)."""
    m = np.zeros((H, W), dtype=bool); used = 0
    for c, cx, cy, bw, bh in boxes:
        if class_id is not None and int(c) != class_id: continue
        x0 = int(np.floor((cx - bw / 2) * W)); x1 = int(np.ceil((cx + bw / 2) * W))
        y0 = int(np.floor((cy - bh / 2) * H)); y1 = int(np.ceil((cy + bh / 2) * H))
        x0, y0, x1, y1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
        if x1 > x0 and y1 > y0: m[y0:y1, x0:x1] = True; used += 1
    return m, used


def centre_prior_224(sigma_frac=0.25, size=224):
    """Fixed Gaussian centred on the network input. Baseline that uses no image information."""
    y, x = np.mgrid[0:size, 0:size]; c = (size - 1) / 2; s = sigma_frac * size
    return np.exp(-((x - c) ** 2 + (y - c) ** 2) / (2 * s * s)).astype(np.float32)


def metrics_for_map(heat, mask, thresholds=(0.3, 0.5, 0.7)):
    """heat: (H,W) in [0,1]; mask: (H,W) bool ground-truth box union. All values are computed here."""
    H, W = heat.shape
    iy, ix = np.unravel_index(int(np.argmax(heat)), heat.shape)
    out = {'hit': bool(mask[iy, ix]), 'peak_x': int(ix), 'peak_y': int(iy), 'box_area_frac': float(mask.mean())}
    for t in thresholds:
        b = heat >= t * heat.max() if heat.max() > 0 else np.zeros_like(mask)
        inter = np.logical_and(b, mask).sum(); union = np.logical_or(b, mask).sum()
        out[f'iou_{t}'] = float(inter / union) if union else float('nan')
        out[f'hot_area_frac_{t}'] = float(b.mean())
    tot = heat.sum()
    out['enrichment'] = float((heat[mask].sum() / tot) / mask.mean()) if tot > 0 and mask.any() else float('nan')
    return out


def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'), float('nan'))
    p = k / n; d = 1 + z * z / n; c = p + z * z / (2 * n); r = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n))
    return ((c - r) / d, (c + r) / d)


def boot_ci(x, stat=np.mean, n=2000, seed=0):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    if len(x) == 0: return (float('nan'), float('nan'))
    rng = np.random.default_rng(seed)
    v = [stat(x[rng.integers(0, len(x), len(x))]) for _ in range(n)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))


def rank_corr(a, b):
    """Spearman correlation between two heatmaps of the same shape."""
    r = spearmanr(np.ravel(a), np.ravel(b)).correlation
    return float(r)

In [ ]:
%%writefile /content/stage8/gradcam.py
"""Grad-CAM for the ResNet18 classifier (PyTorch). Written for Colab; keep the class API stable, the analysis script uses it.

Grad-CAM (Selvaraju et al., 2017): for a target class c and a convolutional layer with activations A_k,
    alpha_k = mean over positions of d(logit_c) / dA_k ;   CAM = ReLU( sum_k alpha_k * A_k )
For ResNet18 the layer4 output goes straight into global average pooling and one linear layer, so
d(logit_c)/dA_k(x,y) = W[c,k] / (H*W) exactly. `analytic_check` verifies the hooked gradients against this identity,
which is an independent test that the hooks capture the right tensor.
"""
import copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
import torchvision.models as tvm
from torchvision import transforms as T

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
CLASS_NAMES = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']


class PadToSquare:
    """Same operation as Stage 2: centred black square canvas."""
    def __call__(self, img):
        w, h = img.size; s = max(w, h)
        if w == h: return img
        canvas = Image.new(img.mode, (s, s), (0,) * len(img.getbands())); canvas.paste(img, ((s - w) // 2, (s - h) // 2)); return canvas


EVAL_TF = T.Compose([PadToSquare(), T.Resize((224, 224)), T.ToTensor(), T.Normalize(MEAN, STD)])


def load_batch(paths):
    return torch.stack([EVAL_TF(Image.open(p).convert('RGB')) for p in paths])


class Explainer:
    def __init__(self, model, device):
        self.model, self.device = model.to(device).eval(), device

    @classmethod
    def from_checkpoint(cls, path, device):
        ck = torch.load(path, map_location=device, weights_only=False)   # our own file from Stage 4/6
        cfg = ck.get('config', {}) or {}
        assert cfg.get('arch', 'resnet18') == 'resnet18' and float(cfg.get('dropout', 0.0)) == 0.0, f'unexpected config {cfg}'
        m = tvm.resnet18(weights=None); m.fc = nn.Linear(512, 4)
        m.load_state_dict(ck['model_state']); return cls(m, device), ck

    def _layer(self, name): return dict(layer4=self.model.layer4, layer3=self.model.layer3)[name]

    def forward_cam(self, paths, layer='layer4', targets=None, batch_size=32):
        """Returns dict(logits, probs, preds, cams(raw, ReLU'd, upsampled to 224 x 224), acts_shape).
        targets=None -> predicted class; otherwise an int array with one target class per image."""
        store = {}
        def fwd_hook(_, __, out):
            store['a'] = out
            out.register_hook(lambda g: store.__setitem__('g', g))
        h = self._layer(layer).register_forward_hook(fwd_hook)
        L, C = [], []
        try:
            for i in range(0, len(paths), batch_size):
                x = load_batch(paths[i:i + batch_size]).to(self.device)
                with torch.enable_grad():
                    logits = self.model(x)
                    t = logits.argmax(1) if targets is None else torch.as_tensor(np.asarray(targets[i:i + batch_size]), device=self.device)
                    self.model.zero_grad(set_to_none=True)
                    logits.gather(1, t[:, None]).sum().backward()      # samples are independent (BatchNorm in eval mode)
                a, g = store['a'].detach(), store['g'].detach()
                alpha = g.mean(dim=(2, 3), keepdim=True)
                cam = torch.relu((alpha * a).sum(1, keepdim=True))
                cam = nn.functional.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[:, 0]
                L.append(logits.detach().cpu()); C.append(cam.cpu())
        finally:
            h.remove()
        logits = torch.cat(L); probs = torch.softmax(logits, 1)
        return dict(logits=logits.numpy(), probs=probs.numpy().astype(np.float64), preds=logits.argmax(1).numpy(),
                    cams=torch.cat(C).numpy().astype(np.float32), acts_shape=tuple(a.shape[1:]))

    def analytic_check(self, paths):
        """Layer4: hooked alpha must equal fc.weight[c] / (H*W); also GAP identity  mean_xy(sum_k W_ck A_k) + b_c = logit_c."""
        store = {}
        def fwd_hook(_, __, out):
            store['a'] = out; out.register_hook(lambda g: store.__setitem__('g', g))
        h = self.model.layer4.register_forward_hook(fwd_hook)
        try:
            x = load_batch(paths).to(self.device)
            with torch.enable_grad():
                logits = self.model(x); t = logits.argmax(1)
                self.model.zero_grad(set_to_none=True); logits.gather(1, t[:, None]).sum().backward()
            a, g = store['a'].detach(), store['g'].detach(); n, k, hh, ww = a.shape
            alpha = g.mean(dim=(2, 3)); Wc = self.model.fc.weight.detach()[t]; b = self.model.fc.bias.detach()[t]
            err_alpha = float((alpha - Wc / (hh * ww)).abs().max()); scale = float((Wc / (hh * ww)).abs().max())
            recon = (Wc[:, :, None, None] * a).sum(1).mean(dim=(1, 2)) + b
            err_logit = float((recon - logits.detach()[torch.arange(n), t]).abs().max())
        finally:
            h.remove()
        return dict(max_abs_alpha_error=err_alpha, alpha_scale=scale, max_abs_logit_reconstruction_error=err_logit, n=int(n))

    def randomised(self, mode, seed=0):
        """Copy with re-initialised weights. mode: 'fc' | 'layer4+fc' | 'all'. Used for the sanity check of Adebayo et al. (2018)."""
        torch.manual_seed(seed)
        m = copy.deepcopy(self.model).cpu()
        parts = {'fc': [m.fc], 'layer4+fc': [m.layer4, m.fc], 'all': [m]}[mode]
        for p in parts:
            for mod in p.modules():
                if hasattr(mod, 'reset_parameters'): mod.reset_parameters()
        return Explainer(m, self.device)

In [ ]:
%%writefile /content/stage8/gradcam_analysis.py
"""Stage 8: Grad-CAM and comparison with the annotated bounding boxes (final model, test set).

Usage: python gradcam_analysis.py --results <final_results> --data-root <Train/Val folder> --ckpt <R18-C_best.pt> --out <dir> [--src <folder with gradcam.py, localisation.py>]

What is measured (all computed here, nothing is typed in):
  * sanity checks: checkpoint reproduces the saved test predictions; hooked gradients equal the analytic ones;
    heatmaps change when the weights are randomised
  * localisation of Grad-CAM peaks and hot regions against the annotated boxes (tumour classes only), with baselines
What is NOT claimed: that the heatmaps show what the network 'uses', or that the model localises tumours. Boxes are an
imperfect reference (Stage 1 found annotation oddities) and the layer4 map is only 7 x 7.
"""
import argparse, json, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

CL = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']; TUMOUR = {'Glioma': 0, 'Meningioma': 1, 'Pituitary': 3}
COL = {'Grad-CAM (layer4)': '#2a78d6', 'Grad-CAM (layer3)': '#1baf7a', 'Centre prior': '#8a8985', 'Permuted heatmap': '#b9b8b3', 'Random point': '#e4e3df'}
INK = '#0b0b0b'


def main():
    ap = argparse.ArgumentParser()
    for k in ['results', 'data_root', 'ckpt', 'out']: ap.add_argument('--' + k.replace('_', '-'), required=True)
    ap.add_argument('--src', default=str(Path(__file__).parent)); ap.add_argument('--device', default='cuda'); ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--n-random-check', type=int, default=60)
    a = ap.parse_args(); sys.path.insert(0, str(Path(__file__).parent)); sys.path.insert(0, a.src)
    import localisation as L
    import gradcam as G
    res, root, out = Path(a.results), Path(a.data_root), Path(a.out)
    for d in ['tables', 'figures']: (out / d).mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False, 'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white'})
    sp = pd.read_csv(res / 'metrics' / 'splits.csv'); te = sp[sp.split == 'test'].reset_index(drop=True)
    paths = [str(root / r) for r in te.rel]; N = len(te); sanity = {}
    ex, ck = G.Explainer.from_checkpoint(a.ckpt, a.device)
    sanity['checkpoint'] = dict(epoch=int(ck.get('epoch', -1)), monitor=ck.get('monitor'), monitor_value=float(ck.get('monitor_value', float('nan'))))

    # ---- 1. forward pass: does this checkpoint reproduce a saved run? ------------------------------------------------
    base = ex.forward_cam(paths, 'layer4'); P = base['probs']; pred = base['preds']
    match = {}
    for cand in ['R18-C', 'R18-C-orig']:
        f = res / 'predictions' / f'{cand}_test_predictions.csv'
        if not f.exists(): continue
        s = pd.read_csv(f).set_index('rel').loc[te.rel]; sp_probs = s[[f'prob_{c}' for c in CL]].values
        match[cand] = dict(label_agreement=float((s.y_pred.values == pred).mean()), max_abs_prob_diff=float(np.abs(sp_probs - P).max()),
                           n_label_disagreements=int((s.y_pred.values != pred).sum()))
    sanity['reproduces_saved_predictions'] = match
    best = min(match, key=lambda k: match[k]['n_label_disagreements']); sanity['matched_run'] = best
    assert match[best]['label_agreement'] >= 0.99, f'checkpoint does not reproduce a saved run: {match}'
    saved = pd.read_csv(res / 'predictions' / f'{best}_test_predictions.csv').set_index('rel').loc[te.rel]
    te['pred'] = pred; te['prob_pred'] = P.max(1); te['correct'] = te.label_idx.values == pred
    print('Checkpoint matches saved run', best, match[best])

    # ---- 2. hook correctness -----------------------------------------------------------------------------------------
    sanity['analytic_gradient_check'] = chk = ex.analytic_check(paths[:16])
    assert chk['max_abs_alpha_error'] < 1e-3 * max(chk['alpha_scale'], 1e-9) + 1e-7 and chk['max_abs_logit_reconstruction_error'] < 1e-3, chk
    print('Analytic gradient check passed:', chk)

    # ---- 3. heatmaps --------------------------------------------------------------------------------------------------
    cam4 = base['cams']; cam3 = ex.forward_cam(paths, 'layer3')['cams']
    err_idx = np.where(~te.correct.values)[0]
    cam4_true = ex.forward_cam([paths[i] for i in err_idx], 'layer4', targets=te.label_idx.values[err_idx])['cams'] if len(err_idx) else np.zeros((0, 224, 224), np.float32)
    flat4 = int((cam4.reshape(N, -1).max(1) == 0).sum()); sanity['images_with_all_zero_layer4_cam'] = flat4

    # ---- 4. localisation ----------------------------------------------------------------------------------------------
    rng = np.random.default_rng(a.seed); rows = []; pad_shares = []
    ev = te[te.label.isin(TUMOUR)].copy(); ev['boxes'] = ev.boxes_json.map(json.loads)
    n_no_match = 0; centre = L.centre_prior_224(); by_class_idx = {c: ev.index[ev.label == c].values for c in TUMOUR}
    perm = {r: {} for r in range(5)}
    for r in range(5):  # five independent draws of a heatmap from ANOTHER test image of the same class
        rr = np.random.default_rng(a.seed + 1 + r)
        for c, idx in by_class_idx.items():
            for i in idx:
                others = idx[idx != i]; perm[r][i] = int(rr.choice(others))
    for i, row in ev.iterrows():
        W, H = int(row.width), int(row.height)
        mask, used = L.box_mask(row.boxes, W, H, class_id=TUMOUR[row.label])
        if used == 0: n_no_match += 1; continue
        rec = dict(rel=row.rel, label=row.label, pred=CL[row.pred], correct=bool(row.correct), prob_pred=row.prob_pred, W=W, H=H, n_boxes_used=used)
        h4, ps = L.to_original(cam4[i], W, H); rec['pad_share_l4'] = ps
        for name, heat in [('l4', h4), ('l3', L.to_original(cam3[i], W, H)[0]), ('centre', L.to_original(centre, W, H)[0])]:
            for k, v in L.metrics_for_map(heat, mask).items(): rec[f'{name}_{k}'] = v
        for r in range(5):
            m = L.metrics_for_map(L.to_original(cam4[perm[r][i]], W, H)[0], mask)
            for k in ['hit', 'iou_0.5', 'enrichment']: rec[f'perm{r}_{k}'] = m[k]
        rec['random_point_hit_expected'] = float(mask.mean())
        rows.append(rec)
    D = pd.DataFrame(rows); D.to_csv(out / 'tables' / 'gradcam_per_image_test.csv', index=False)
    sanity['tumour_test_images'] = int(len(ev)); sanity['tumour_images_without_matching_class_box'] = int(n_no_match); sanity['evaluated_images'] = int(len(D))

    def summ(d, tag):
        n = len(d); out_rows = []
        for name, pre in [('Grad-CAM (layer4)', 'l4'), ('Grad-CAM (layer3)', 'l3'), ('Centre prior', 'centre')]:
            k = int(d[f'{pre}_hit'].sum()); lo, hi = L.wilson(k, n)
            out_rows.append(dict(group=tag, method=name, n=n, pointing_hits=k, pointing_rate=k / n, pointing_ci_low=lo, pointing_ci_high=hi,
                                 iou_0_5_mean=d[f'{pre}_iou_0.5'].mean(), iou_0_5_ci_low=L.boot_ci(d[f'{pre}_iou_0.5'])[0], iou_0_5_ci_high=L.boot_ci(d[f'{pre}_iou_0.5'])[1],
                                 iou_0_3_mean=d[f'{pre}_iou_0.3'].mean(), iou_0_7_mean=d[f'{pre}_iou_0.7'].mean(),
                                 enrichment_median=d[f'{pre}_enrichment'].median(), enrichment_mean=d[f'{pre}_enrichment'].mean(),
                                 enrichment_ci_low=L.boot_ci(d[f'{pre}_enrichment'])[0], enrichment_ci_high=L.boot_ci(d[f'{pre}_enrichment'])[1]))
        ph = [d[f'perm{r}_hit'].mean() for r in range(5)]; pi = [d[f'perm{r}_iou_0.5'].mean() for r in range(5)]; pe = [d[f'perm{r}_enrichment'].mean() for r in range(5)]
        out_rows.append(dict(group=tag, method='Permuted heatmap', n=n, pointing_rate=np.mean(ph), pointing_ci_low=min(ph), pointing_ci_high=max(ph),
                             iou_0_5_mean=np.mean(pi), iou_0_5_ci_low=min(pi), iou_0_5_ci_high=max(pi), enrichment_mean=np.mean(pe), enrichment_ci_low=min(pe), enrichment_ci_high=max(pe)))
        out_rows.append(dict(group=tag, method='Random point', n=n, pointing_rate=d.random_point_hit_expected.mean()))
        return out_rows
    S = pd.DataFrame(summ(D, 'all evaluated tumour images'))
    for c in TUMOUR: S = pd.concat([S, pd.DataFrame(summ(D[D.label == c], c))])
    S = pd.concat([S, pd.DataFrame(summ(D[D.correct], 'correctly classified')), pd.DataFrame(summ(D[~D.correct], 'misclassified'))] if (~D.correct).any() else [S, pd.DataFrame(summ(D[D.correct], 'correctly classified'))])
    D['size_tercile'] = pd.qcut(D.l4_box_area_frac, 3, labels=['small boxes', 'medium boxes', 'large boxes'])
    for t, d in D.groupby('size_tercile', observed=True): S = pd.concat([S, pd.DataFrame(summ(d, f'{t} (area {d.l4_box_area_frac.min():.3f} to {d.l4_box_area_frac.max():.3f} of image)'))])
    S.to_csv(out / 'tables' / 'gradcam_localisation_summary.csv', index=False)
    print(S[S.group == 'all evaluated tumour images'][['method', 'n', 'pointing_rate', 'pointing_ci_low', 'pointing_ci_high', 'iou_0_5_mean', 'enrichment_median']].round(3).to_string(index=False))

    # ---- 5. weight-randomisation sanity check -------------------------------------------------------------------------
    ev_pos = ev.index.values; sel = rng.choice(ev_pos, size=min(a.n_random_check, len(ev_pos)), replace=False)
    sel_paths = [paths[i] for i in sel]; trained = cam4[sel]; rand_rows = []; rand_cams = {}
    for mode in ['fc', 'layer4+fc', 'all']:
        rx = ex.randomised(mode, seed=a.seed); r_cam = rx.forward_cam(sel_paths, 'layer4', targets=pred[sel])['cams']; rand_cams[mode] = r_cam
        rho = [L.rank_corr(trained[j], r_cam[j]) if r_cam[j].max() > 0 and trained[j].max() > 0 else float('nan') for j in range(len(sel))]
        hit = []
        for j, i in enumerate(sel):
            row = te.loc[i]; mask, used = L.box_mask(json.loads(row.boxes_json), int(row.width), int(row.height), class_id=TUMOUR[row.label])
            if used: hit.append(L.metrics_for_map(L.to_original(r_cam[j], int(row.width), int(row.height))[0], mask)['hit'])
        rand_rows.append(dict(randomised=mode, n_images=len(sel), mean_spearman_with_trained_map=float(np.nanmean(rho)), median_spearman=float(np.nanmedian(rho)),
                              pointing_rate_randomised=float(np.mean(hit)), all_zero_maps=int(sum(r_cam[j].max() == 0 for j in range(len(sel))))))
    R = pd.DataFrame(rand_rows); R.to_csv(out / 'tables' / 'gradcam_randomisation_check.csv', index=False); sanity['randomisation_check'] = rand_rows
    print(R.round(3).to_string(index=False))

    # ---- 6. figures ---------------------------------------------------------------------------------------------------
    from PIL import Image
    def show(ax, i, cam, title, boxes=True):
        row = te.loc[i]; img = Image.open(root / row.rel).convert('L'); W, H = img.size
        heat = L.to_original(cam, W, H)[0]
        ax.imshow(img, cmap='gray', vmin=0, vmax=255); ax.imshow(heat, cmap='magma', alpha=0.45, vmin=0, vmax=1)
        if heat.max() > 0: ax.contour(heat, levels=[0.5], colors='white', linewidths=0.8, linestyles='dotted')
        if boxes:
            mask, _ = L.box_mask(json.loads(row.boxes_json), W, H, class_id=TUMOUR.get(row.label, -1))
            if mask.any(): ax.contour(mask.astype(float), levels=[0.5], colors='#7fe3ff', linewidths=1.4)
        ax.set_title(title, fontsize=8, loc='left'); ax.axis('off')
    def grid(items, path, ncols=4, suptitle=''):
        nr = int(np.ceil(len(items) / ncols)); fig, axs = plt.subplots(nr, ncols, figsize=(3.2 * ncols, 3.5 * nr)); axs = np.atleast_2d(axs)
        for ax in axs.ravel(): ax.axis('off')
        for ax, (i, cam, t) in zip(axs.ravel(), items): show(ax, i, cam, t)
        fig.tight_layout(h_pad=2.2, rect=[0, 0, 1, 0.97]); fig.suptitle(suptitle, fontsize=9, x=0.01, y=0.995, ha='left'); fig.savefig(path, dpi=170, bbox_inches='tight'); plt.close(fig)
    cap = 'cyan outline = annotated box; dotted white = 50% contour of the heatmap; heat is min-max normalised per image'
    ok = D[D.correct]; items = []
    for c in TUMOUR:
        pick = ok[ok.label == c].sample(min(4, (ok.label == c).sum()), random_state=a.seed)
        for _, r in pick.iterrows():
            i = int(te.index[te.rel == r.rel][0]); items.append((i, cam4[i], f'{r.label}: correct, p={r.prob_pred:.2f}\npeak {"inside" if r.l4_hit else "outside"} box'))
    grid(items, out / 'figures' / 'gradcam_random_correct_examples.png', suptitle='Random correctly classified test images (4 per tumour class, fixed seed). ' + cap)
    items = []
    for j, i in enumerate(err_idx):
        row = te.loc[i]; items.append((int(i), cam4[i], f'TRUE {row.label} -> PRED {CL[row.pred]} (p={row.prob_pred:.2f})\nmap for predicted class'))
        if row.label in TUMOUR: items.append((int(i), cam4_true[j], f'same image, map for TRUE class {row.label}'))
    if items: grid(items, out / 'figures' / 'gradcam_misclassified_examples.png', suptitle='All misclassified test images of the final model. ' + cap)
    # bar chart vs baselines
    A = S[S.group == 'all evaluated tumour images'].set_index('method'); order = ['Grad-CAM (layer4)', 'Grad-CAM (layer3)', 'Centre prior', 'Permuted heatmap', 'Random point']
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    v = A.loc[order, 'pointing_rate'].values * 100; lo = A.loc[order, 'pointing_ci_low'].values * 100; hi = A.loc[order, 'pointing_ci_high'].values * 100
    for k, m in enumerate(order):
        ax[0].barh(k, v[k], color=COL[m]); 
        if not np.isnan(lo[k]): ax[0].plot([lo[k], hi[k]], [k, k], color=INK, lw=1.2)
        ax[0].text(max(v[k], hi[k] if not np.isnan(hi[k]) else 0) + 1, k, f'{v[k]:.1f}%', va='center')
    ax[0].set_yticks(range(len(order))); ax[0].set_yticklabels(order); ax[0].invert_yaxis(); ax[0].set_xlim(0, 105)
    ax[0].set_xlabel('Pointing game: peak of heatmap inside an annotated box (%)'); ax[0].set_title(f'n = {len(D)} tumour test images', loc='left')
    iv = A.loc[order[:4], 'iou_0_5_mean'].values; il = A.loc[order[:4], 'iou_0_5_ci_low'].values; ih = A.loc[order[:4], 'iou_0_5_ci_high'].values
    for k, m in enumerate(order[:4]):
        ax[1].barh(k, iv[k], color=COL[m]); ax[1].plot([il[k], ih[k]], [k, k], color=INK, lw=1.2); ax[1].text(ih[k] + .005, k, f'{iv[k]:.3f}', va='center')
    ax[1].set_yticks(range(4)); ax[1].set_yticklabels(order[:4]); ax[1].invert_yaxis(); ax[1].set_xlabel('Mean IoU of hot region (>= 50% of peak) with box'); ax[1].set_xlim(0, max(ih.max() * 1.25, .1))
    ax[1].set_title('whiskers: 95% CI (Wilson / bootstrap); permuted: range of 5 draws', loc='left', fontsize=9)
    fig.tight_layout(); fig.savefig(out / 'figures' / 'gradcam_localisation_vs_baselines.png', dpi=200, bbox_inches='tight'); plt.close(fig)
    # by class and size
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for axx, col, ttl in [(ax[0], 'label', 'By tumour class'), (ax[1], 'size_tercile', 'By box size (terciles of box area)')]:
        gs = D.groupby(col, observed=True); xs = np.arange(len(gs)); w = .38
        for o, (pre, nm) in enumerate([('l4', 'Grad-CAM (layer4)'), ('centre', 'Centre prior')]):
            rates = [g[f'{pre}_hit'].mean() * 100 for _, g in gs]; axx.bar(xs + (o - .5) * w, rates, w, color=COL[nm], label=nm)
            for x_, r_, (_, g) in zip(xs + (o - .5) * w, rates, gs): axx.text(x_, r_ + 1, f'{r_:.0f}%', ha='center', fontsize=8)
        axx.set_xticks(xs); axx.set_xticklabels([f'{k}\n(n={len(g)})' for k, g in gs]); axx.set_ylim(0, 112); axx.set_ylabel('Pointing hit rate (%)'); axx.set_title(ttl, loc='left')
    ax[0].legend(loc='lower left'); fig.tight_layout(); fig.savefig(out / 'figures' / 'gradcam_localisation_by_class_and_size.png', dpi=200, bbox_inches='tight'); plt.close(fig)
    # randomisation figure
    show_idx = sel[:3]; fig, axs = plt.subplots(len(show_idx), 4, figsize=(12.5, 3.3 * len(show_idx))); axs = np.atleast_2d(axs)
    for r_, i in enumerate(show_idx):
        show(axs[r_, 0], int(i), cam4[i], f'trained model ({te.loc[i].label})')
        for c_, mode in enumerate(['fc', 'layer4+fc', 'all']): show(axs[r_, c_ + 1], int(i), rand_cams[mode][list(sel).index(i)], f'randomised: {mode}')
    fig.tight_layout(rect=[0, 0, 1, 0.96]); fig.suptitle('Sanity check: heatmaps should change when weights are randomised', fontsize=10, x=0.01, y=0.995, ha='left'); fig.savefig(out / 'figures' / 'gradcam_randomisation_check.png', dpi=170, bbox_inches='tight'); plt.close(fig)
    sanity['mean_heat_share_on_padding_layer4'] = float(np.nanmean(D.pad_share_l4)); sanity['note'] = 'Boxes used: only boxes whose class id equals the image class. No Tumor images are excluded (their boxes cover most of the brain).'
    json.dump(sanity, open(out / 'gradcam_sanity_checks.json', 'w'), indent=1, default=float)
    print('\nDONE. Files in', out)


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/stage8/test_localisation.py
"""Self-tests with synthetic heatmaps; run: python3 test_localisation.py"""
import numpy as np
from localisation import *
# 1. mapping round trip on square and non-square images: a peak painted at a known place in the 224 frame
for (W, H) in [(512, 512), (201, 251), (630, 590)]:
    s, ox, oy = pad_offset(W, H)
    # choose a target pixel in the original image, find its location in the 224 frame
    tx, ty = int(W * 0.7), int(H * 0.3)
    px, py = int((tx + ox) / s * 224), int((ty + oy) / s * 224)
    cam = np.zeros((224, 224), np.float32); cam[max(py - 2, 0):py + 3, max(px - 2, 0):px + 3] = 1
    m, pad = to_original(cam, W, H)
    iy, ix = np.unravel_index(m.argmax(), m.shape)
    err = max(abs(ix - tx), abs(iy - ty)); tol = s / 224 * 3
    print(f'{W}x{H}: target ({tx},{ty}) -> peak ({ix},{iy}), error {err}px, tolerance {tol:.1f}px, pad share {pad:.3f}')
    assert err <= tol and m.shape == (H, W) and 0 <= m.min() and m.max() == 1
# 2. a heatmap that is only on padding gives a nearly flat crop and a large pad share
cam = np.zeros((224, 224), np.float32); cam[:, :5] = 1
m, pad = to_original(cam, 201, 251); print('padding-only heat: pad share', round(pad, 3)); assert pad > .9
# 3. box mask
boxes = [[0, .5, .5, .2, .1], [3, .1, .1, .1, .1]]
mk, n = box_mask(boxes, 200, 100, class_id=0); assert n == 1 and abs(mk.mean() - .02) < .005, (n, mk.mean())
mk_all, n2 = box_mask(boxes, 200, 100); assert n2 == 2
# 4. metrics: perfect, disjoint and uniform heatmaps
H, W = 100, 200; mask, _ = box_mask([[0, .5, .5, .2, .2]], W, H)
heat = np.zeros((H, W)); heat[mask] = 1; r = metrics_for_map(heat, mask); assert r['hit'] and abs(r['iou_0.5'] - 1) < 1e-9 and r['enrichment'] > 4
heat2 = np.zeros((H, W)); heat2[:10, :10] = 1; r2 = metrics_for_map(heat2, mask); assert not r2['hit'] and r2['iou_0.5'] == 0
heat3 = np.ones((H, W)); r3 = metrics_for_map(heat3, mask); assert abs(r3['enrichment'] - 1) < 1e-9
# 5. random heatmaps hit boxes at about the box-area fraction
rng = np.random.default_rng(0); hits = [metrics_for_map(rng.random((H, W)), mask)['hit'] for _ in range(4000)]
print('random heatmap hit rate', np.mean(hits), 'expected', mask.mean()); assert abs(np.mean(hits) - mask.mean()) < .012
# 6. centre prior is symmetric and peaks in the middle
c = centre_prior_224(); assert np.unravel_index(c.argmax(), c.shape) in [(111, 111), (111, 112), (112, 111), (112, 112)]
# 7. stats helpers
lo, hi = wilson(50, 100); assert .40 < lo < .41 and .59 < hi < .60
print('boot', boot_ci(np.arange(100)), 'spearman', rank_corr(np.arange(50), np.arange(50)))
print('ALL LOCALISATION TESTS PASSED')

## 2. Test the geometry and metric code
Synthetic heatmaps with known answers (no model involved). Must end with `ALL LOCALISATION TESTS PASSED`.

In [ ]:
!cd /content/stage8 && python test_localisation.py

## 3. Inputs

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
CKPT = Path('/content/drive/MyDrive/brain-tumor-project/models/R18-C_best.pt')
assert CKPT.exists(), f'Not found: {CKPT}. Open the models folder in Drive and check the file name.'
print('checkpoint:', CKPT, round(CKPT.stat().st_size/1e6, 1), 'MB')
RES = Path('/content/final_results')
if not (RES/'metrics'/'splits.csv').exists():
    from google.colab import files
    print('Upload final_results.zip'); up = files.upload()
    with zipfile.ZipFile(list(up)[0]) as f: f.extractall(RES)
    if not (RES/'metrics').exists():
        sub = [p for p in RES.iterdir() if (p/'metrics').exists()]; assert sub; RES = sub[0]
print('results:', RES)

In [ ]:
DATA_SOURCE = 'kaggle'                                         # 'kaggle' or 'drive'
DRIVE_ZIP = '/content/drive/MyDrive/brain_tumor_dataset.zip'   # only used if DATA_SOURCE == 'drive'
SLUG = 'ahmedsorour1/mri-for-brain-tumor-with-bounding-boxes'
RAW = Path('/content/data_raw'); RAW.mkdir(exist_ok=True)
if not any(p.is_file() for p in RAW.rglob('*')):
    if DATA_SOURCE == 'drive':
        assert Path(DRIVE_ZIP).exists(), f'Not found: {DRIVE_ZIP}'
        with zipfile.ZipFile(DRIVE_ZIP) as f: f.extractall(RAW)
    else:
        if not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
            try:
                from google.colab import userdata
                os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            except Exception:
                from google.colab import files
                print('Upload kaggle.json'); up = files.upload()
                kd = Path.home()/'.kaggle'; kd.mkdir(exist_ok=True); (kd/'kaggle.json').write_bytes(up[list(up)[0]]); os.chmod(kd/'kaggle.json', 0o600)
        subprocess.run(['pip','-q','install','kaggle'], check=True)
        r = subprocess.run(['kaggle','datasets','download','-d',SLUG,'-p',str(RAW),'--unzip'], capture_output=True, text=True)
        print(r.stdout[-500:], r.stderr[-800:]); r.check_returncode()
def find_root(base):
    for p in [base] + [q for q in base.rglob('*') if q.is_dir()]:
        if {'train','val'} <= {c.name.lower() for c in p.iterdir() if c.is_dir()}: return p
    raise FileNotFoundError('Train/ and Val/ not found')
ROOT = find_root(RAW); print('Dataset root:', ROOT)

## 4. Run the analysis
The run stops with an error if (a) the checkpoint does not reproduce a saved run (at least 99 percent label agreement, i.e. at most 5 of 544 labels differ; small differences can come from GPU nondeterminism), (b) the hooked gradients differ from the exact formula, so no heatmap is produced from a broken setup.
The printed lines to look for: `Checkpoint matches saved run ...` and `Analytic gradient check passed`.

In [ ]:
OUT = Path('/content/stage8_out'); OUT.mkdir(exist_ok=True)
!python /content/stage8/gradcam_analysis.py --results {RES} --data-root {ROOT} --ckpt {CKPT} --out {OUT} --src /content/stage8 --device cuda

## 5. Look at the results

In [ ]:
from IPython.display import Image, display
import pandas as pd; pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
for f in ['gradcam_localisation_vs_baselines','gradcam_localisation_by_class_and_size','gradcam_random_correct_examples','gradcam_misclassified_examples','gradcam_randomisation_check']:
    p = OUT/'figures'/f'{f}.png'
    if p.exists(): display(Image(str(p), width=950))
S = pd.read_csv(OUT/'tables'/'gradcam_localisation_summary.csv')
print(S[S.group=='all evaluated tumour images'][['method','n','pointing_rate','pointing_ci_low','pointing_ci_high','iou_0_5_mean','enrichment_mean']].round(3).to_string(index=False))
print(pd.read_csv(OUT/'tables'/'gradcam_randomisation_check.csv').round(3).to_string(index=False))
print(json.load(open(OUT/'gradcam_sanity_checks.json'))['reproduces_saved_predictions'])

## 6. Package
Send `stage8_results.zip` back.

In [ ]:
import shutil
shutil.copy('/content/stage8/gradcam_analysis.py', OUT); shutil.copy('/content/stage8/gradcam.py', OUT); shutil.copy('/content/stage8/localisation.py', OUT)
shutil.make_archive('/content/stage8_results', 'zip', OUT)
from google.colab import files; files.download('/content/stage8_results.zip')